In [3]:
!pip install transformers torch -q

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)  # should print "cuda" — if not, go to Runtime > Change runtime type > GPU

cpu


In [4]:
target_name = "gpt2-medium"
draft_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(target_name)
target_model = AutoModelForCausalLM.from_pretrained(target_name).to(device)
draft_model = AutoModelForCausalLM.from_pretrained(draft_name).to(device)

config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [15]:
from collections import defaultdict

free_blocks = [i for i in range(20)]
block_table = defaultdict(list)

def allocate_block(sequence_id):
    if not free_blocks:
        print("Error: No free blocks available")
        return

    block = free_blocks.pop()
    block_table[sequence_id].append(block)

    print(f"Block {block} added to sequence_id {sequence_id}")
    return

def free_sequence(sequence_id):
    global free_blocks
    if sequence_id not in block_table:
        print("Error: sequence_id not in block table")
        return


    freed_blocks = block_table[sequence_id]
    del block_table[sequence_id]
    free_blocks += freed_blocks
    print("Freed blocks:", freed_blocks)
    return


In [ ]:
allocate_block("seq_A")
allocate_block("seq_A")
allocate_block("seq_B")
print(free_blocks)
print(block_table)

free_sequence("seq_A")
print(free_blocks)
print(block_table)

In [16]:
# Integrating blocks into decode
import math
prompt = "The weather today is"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

sequence_id = "seq_1"
num_prompt_blocks = math.ceil(input_ids.shape[1]/4)
for _ in range(num_prompt_blocks):
  allocate_block(sequence_id)

past_key_values = None
current_token = input_ids
generated = input_ids
num_new_tokens = 12
print("Prompt Length", input_ids.shape[1])

for step in range(num_new_tokens):
  with torch.no_grad():
    outputs = target_model(current_token, past_key_values=past_key_values, use_cache=True)
  next_token_logits = outputs.logits[0, -1, :]
  next_token_id = torch.argmax(next_token_logits).unsqueeze(0).unsqueeze(0)

  if generated.shape[1] % 4 == 0:
    allocate_block(sequence_id)

  generated = torch.cat([generated, next_token_id], dim=1)
  past_key_values = outputs.past_key_values
  current_token = next_token_id

free_sequence(sequence_id)

Block 19 added to sequence_id seq_1
Prompt Length 4
Block 18 added to sequence_id seq_1
Block 17 added to sequence_id seq_1
Block 16 added to sequence_id seq_1
Freed blocks: [19, 18, 17, 16]
